# 03 — Comprehensive dissertation analysis

**Project:** *The Linguistic Influence Matrix: How Wording Changes LLM Responses*

This notebook analyses `analysis_ready_responses.csv` against the dissertation's three research questions:

1. How do semantically equivalent linguistic prompt variations influence behavioural consistency in LLM-generated responses?
2. Which linguistic prompt structures most significantly affect semantic divergence, hallucination susceptibility and recommendation drift?
3. How does behavioural drift caused by linguistic variation differ across multiple conversational LLMs?

It produces reproducible EDA, NLP-derived measures, paired inferential tests, effect sizes, corrected post-hoc tests, robustness analyses, the Linguistic Influence Matrix, a manual-validation sample, dissertation figures, result tables and a writing guide.

**Important interpretation rule:** the dataset contains observed outputs generated under a fixed 1,200-token ceiling. Automated claim-risk and recommendation measures are explicitly labelled as proxies; they are not treated as verified hallucinations or perfect entity extraction.


## Step 1 — Install and import the analysis stack

The notebook uses a fixed random seed. It installs a package only if Colab does not already provide it.


In [ ]:
import importlib.util
import json
import math
import os
import re
import subprocess
import sys
import warnings
import zipfile
from itertools import combinations
from pathlib import Path

required_packages = {
    "numpy": "numpy>=1.26,<3",
    "pandas": "pandas>=2.2,<4",
    "scipy": "scipy>=1.12,<2",
    "sklearn": "scikit-learn>=1.4,<2",
    "statsmodels": "statsmodels>=0.14,<1",
    "matplotlib": "matplotlib>=3.8,<4",
    "seaborn": "seaborn>=0.13,<1",
}
missing_packages = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics import pairwise_distances
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_SEED = 42
ALPHA = 0.05
rng = np.random.default_rng(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid", context="notebook", palette="colorblind")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 220, "axes.titleweight": "bold"})

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)


## Step 2 — Locate the cleaned dataset and create output folders

Recommended Colab use: put `analysis_ready_responses.csv` directly in My Drive. If it is not found, Colab opens an upload box. The analysis writes into `My Drive/dissertation_analysis_outputs`.


In [ ]:
IN_COLAB = "google.colab" in sys.modules
USE_GOOGLE_DRIVE = True
FILE_NAME = "analysis_ready_responses.csv"

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

input_override = os.environ.get("ANALYSIS_READY_FILE", "").strip()
candidates = [
    Path(input_override) if input_override else None,
    Path("/content/drive/MyDrive") / FILE_NAME,
    Path("/content") / FILE_NAME,
    Path.cwd() / FILE_NAME,
]
INPUT_FILE = next((path for path in candidates if path is not None and path.exists()), None)

if INPUT_FILE is None and IN_COLAB:
    from google.colab import files
    print(f"Please upload {FILE_NAME}")
    uploaded = files.upload()
    if FILE_NAME not in uploaded:
        raise FileNotFoundError(f"The uploaded file must be named {FILE_NAME}")
    INPUT_FILE = Path("/content") / FILE_NAME

if INPUT_FILE is None:
    raise FileNotFoundError(
        f"Could not find {FILE_NAME}. Put it in My Drive, upload it to Colab, "
        "or set ANALYSIS_READY_FILE to its full path."
    )

output_override = os.environ.get("DISSERTATION_ANALYSIS_OUTPUT", "").strip()
if output_override:
    OUTPUT_DIR = Path(output_override)
elif IN_COLAB and USE_GOOGLE_DRIVE:
    OUTPUT_DIR = Path("/content/drive/MyDrive/dissertation_analysis_outputs")
else:
    OUTPUT_DIR = Path.cwd() / "dissertation_analysis_outputs"

TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
VALIDATION_DIR = OUTPUT_DIR / "manual_validation"
for directory in [OUTPUT_DIR, TABLE_DIR, FIGURE_DIR, VALIDATION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Input:", INPUT_FILE)
print("Output:", OUTPUT_DIR)


## Step 3 — Load, standardise and validate the experimental design

The actual study comprises three model versions, five domains, 100 base intents and 10 phrasing conditions. The CSV contains 2,984 observed responses because 16 ChatGPT responses were not collected. Token-ceiling responses are retained and flagged because they are part of the realised experimental output.


In [ ]:
MODEL_ORDER = ["ChatGPT", "Claude", "Gemini"]
DOMAIN_ORDER = ["Technology", "Finance", "Healthcare", "Marketing", "Sports"]
PHRASING_ORDER = [
    "Direct Question", "Conversational", "Formal/Professional", "Informal", "Conditional",
    "Highly Specific", "Ambiguous", "Comparative", "Constraint-Based", "Role-Based/Expert Framing",
]

data = pd.read_csv(INPUT_FILE, low_memory=False)
required_columns = {
    "response_id", "prompt_id", "base_intent_id", "domain", "intent_type", "phrasing_type",
    "prompt_text", "model_name", "model_version", "response_text", "response_word_count",
    "lexical_diversity", "at_output_token_limit", "has_url", "contains_numeric_claim",
    "contains_recommendation_language", "contains_refusal_language", "contains_uncertainty_language",
}
missing_required = sorted(required_columns - set(data.columns))
if missing_required:
    raise KeyError(f"Missing required columns: {missing_required}")

boolean_columns = [
    "search_web_enabled", "response_valid", "has_url", "contains_bulleted_list",
    "contains_numbered_list", "contains_heading", "contains_table", "contains_code",
    "contains_recommendation_language", "contains_refusal_language",
    "contains_uncertainty_language", "contains_numeric_claim", "response_ends_cleanly",
    "at_output_token_limit", "possible_abrupt_truncation", "duplicate_prompt_model_run",
    "duplicate_response_text", "has_collection_limitation",
    "analysis_eligible_without_limitation",
]
for column in boolean_columns:
    if column in data:
        data[column] = data[column].astype(str).str.strip().str.lower().map({"true": True, "false": False}).fillna(False)

numeric_columns = [
    "variation_no", "run_number", "input_tokens", "output_tokens", "response_character_count",
    "response_word_count", "response_sentence_count", "response_paragraph_count",
    "unique_word_count", "lexical_diversity", "url_count",
]
for column in numeric_columns:
    if column in data:
        data[column] = pd.to_numeric(data[column], errors="coerce")

data["model_name"] = pd.Categorical(data["model_name"], categories=MODEL_ORDER, ordered=True)
data["domain"] = pd.Categorical(data["domain"], categories=DOMAIN_ORDER, ordered=True)
data["phrasing_type"] = pd.Categorical(data["phrasing_type"], categories=PHRASING_ORDER, ordered=True)
data["response_text"] = data["response_text"].fillna("").astype(str)

assert data["response_id"].is_unique, "response_id must be unique"
assert data.duplicated(["prompt_id", "model_name", "run_number"]).sum() == 0
assert not data["response_text"].str.strip().eq("").any()
assert set(data["model_name"].dropna().astype(str).unique()) == set(MODEL_ORDER)
assert data["base_intent_id"].nunique() == 100
assert data["prompt_id"].nunique() == 1000

print("Rows:", f"{len(data):,}")
print("Prompts:", data["prompt_id"].nunique())
print("Base intents:", data["base_intent_id"].nunique())
display(data.head(5))


## Step 4 — Data quality, missingness and censoring audit

This audit reconstructs the expected 3,000-row prompt-by-model grid, identifies the 16 absent responses, and quantifies the 1,200-token ceiling by model, domain and phrasing. These are collection limitations, not cleaning failures.


In [ ]:
prompt_metadata = data.sort_values("prompt_id").drop_duplicates("prompt_id")[[
    "prompt_id", "base_intent_id", "variation_no", "domain", "intent_type", "phrasing_type", "prompt_text"
]]
expected_grid = prompt_metadata.assign(_key=1).merge(
    pd.DataFrame({"model_name": MODEL_ORDER, "_key": 1}), on="_key", how="outer"
).drop(columns="_key")
observed_keys = data[["prompt_id", "model_name"]].copy()
observed_keys["model_name"] = observed_keys["model_name"].astype(str)
expected_grid = expected_grid.merge(observed_keys.assign(observed=True), on=["prompt_id", "model_name"], how="left")
expected_grid["missing_response"] = expected_grid["observed"].isna()

coverage_by_model = pd.DataFrame({
    "model_name": MODEL_ORDER,
    "expected_responses": [1000] * len(MODEL_ORDER),
}).merge(
    data.groupby("model_name", observed=True).agg(
        observed_responses=("response_id", "size"),
        token_ceiling_responses=("at_output_token_limit", "sum"),
        abrupt_at_ceiling=("possible_abrupt_truncation", "sum"),
    ).reset_index(), on="model_name", how="left"
)
coverage_by_model["missing_responses"] = coverage_by_model["expected_responses"] - coverage_by_model["observed_responses"]
coverage_by_model["coverage_rate"] = coverage_by_model["observed_responses"] / coverage_by_model["expected_responses"]
coverage_by_model["token_ceiling_rate_observed"] = coverage_by_model["token_ceiling_responses"] / coverage_by_model["observed_responses"]

coverage_by_phrasing = data.groupby(["model_name", "phrasing_type"], observed=True).agg(
    n=("response_id", "size"),
    token_ceiling_n=("at_output_token_limit", "sum"),
).reset_index()
coverage_by_phrasing["token_ceiling_rate"] = coverage_by_phrasing["token_ceiling_n"] / coverage_by_phrasing["n"]

missing_responses = expected_grid.loc[expected_grid["missing_response"]].drop(columns="observed")
quality_summary = pd.DataFrame([
    {"measure": "Expected prompt-model responses", "value": 3000},
    {"measure": "Observed valid responses", "value": int(len(data))},
    {"measure": "Missing responses", "value": int(expected_grid["missing_response"].sum())},
    {"measure": "Responses at 1,200-token ceiling", "value": int(data["at_output_token_limit"].sum())},
    {"measure": "Unique prompt IDs", "value": int(data["prompt_id"].nunique())},
    {"measure": "Unique base intents", "value": int(data["base_intent_id"].nunique())},
    {"measure": "Duplicate prompt-model-run keys", "value": int(data.duplicated(["prompt_id", "model_name", "run_number"]).sum())},
])

quality_summary.to_csv(TABLE_DIR / "data_quality_summary.csv", index=False)
coverage_by_model.to_csv(TABLE_DIR / "coverage_by_model.csv", index=False)
coverage_by_phrasing.to_csv(TABLE_DIR / "coverage_and_ceiling_by_model_phrasing.csv", index=False)
missing_responses.to_csv(TABLE_DIR / "missing_responses_reconstructed.csv", index=False)

display(coverage_by_model)
display(missing_responses.groupby(["model_name", "phrasing_type"], observed=True).size().rename("missing_n").reset_index())


## Step 5 — Derive transparent NLP and behavioural measures

### Measures used

- **Latent-semantic divergence:** TF-IDF word/bigram representations reduced with Latent Semantic Analysis (Truncated SVD), followed by cosine distance from the within-model, within-intent centroid.
- **Lexical instability:** mean pairwise Jaccard distance between content-word sets within each model-intent block.
- **Length deviation:** absolute response-word deviation from the within-block median, divided by that median.
- **Structural deviation:** mismatch from the within-block modal response structure.
- **Recommendation instability proxy:** mean Jaccard distance between heuristically extracted list-item labels; missing when extraction coverage is inadequate.
- **Claim-verification risk proxy:** a transparent rule-based score for unsourced numeric or entity-dense claims. It is not a factuality judgement.

The core behavioural sensitivity score uses semantic, lexical, length and structural components only. Every component remains available separately.


In [ ]:
TOKEN_PATTERN = re.compile(r"\b[A-Za-z][A-Za-z0-9'’\-]{1,}\b")
ITEM_LINE_PATTERN = re.compile(r"(?m)^\s*(?:[-*•]|\d+[.)])\s+(.+)$")
GENERIC_ITEM_LABELS = {
    "summary", "overview", "pros", "cons", "advantages", "disadvantages", "benefits",
    "limitations", "considerations", "recommendation", "recommendations", "conclusion",
    "next steps", "key points", "why", "how", "example", "examples", "factors", "risks",
}


def content_tokens(text):
    return {
        token.casefold() for token in TOKEN_PATTERN.findall(text)
        if token.casefold() not in ENGLISH_STOP_WORDS and len(token) > 2
    }


def extract_recommendation_items(text):
    items = []
    for raw in ITEM_LINE_PATTERN.findall(text):
        candidate = re.split(r"\s+(?:[-–—:]|\(|\bis\b|\bfor\b)\s*", raw.strip(), maxsplit=1)[0]
        candidate = re.sub(r"[*_`#]", "", candidate).strip(" .:;,-–—")
        words = candidate.split()
        if 1 <= len(words) <= 8 and 2 <= len(candidate) <= 70 and candidate.casefold() not in GENERIC_ITEM_LABELS:
            items.append(candidate.casefold())
    return sorted(set(items))


def proper_noun_count(text):
    candidates = re.findall(r"\b[A-Z][A-Za-z0-9&.\-]+(?:\s+[A-Z][A-Za-z0-9&.\-]+){0,2}\b", text)
    stop = {"The", "This", "That", "These", "Those", "However", "Therefore", "For", "In", "On", "A", "An", "I"}
    return len({item for item in candidates if item not in stop})


data["content_token_set"] = data["response_text"].map(content_tokens)
data["recommendation_items"] = data["response_text"].map(extract_recommendation_items)
data["recommendation_item_count"] = data["recommendation_items"].map(len)
data["proper_noun_count_proxy"] = data["response_text"].map(proper_noun_count)

data["claim_verification_risk_proxy"] = (
    0.45 * data["contains_numeric_claim"].astype(float)
    + 0.25 * (data["contains_numeric_claim"] & ~data["has_url"]).astype(float)
    + 0.15 * ((data["proper_noun_count_proxy"] >= 5) & ~data["has_url"]).astype(float)
    + 0.15 * (data["contains_numeric_claim"] & ~data["contains_uncertainty_language"]).astype(float)
).clip(0, 1)

vectorizer = TfidfVectorizer(
    stop_words="english", ngram_range=(1, 2), min_df=2, max_df=0.98,
    max_features=20_000, sublinear_tf=True,
)
tfidf = vectorizer.fit_transform(data["response_text"])
n_components = int(min(200, tfidf.shape[0] - 1, tfidf.shape[1] - 1))
svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_SEED, n_iter=7)
semantic_vectors = normalize(svd.fit_transform(tfidf))
vector_lookup = dict(zip(data["response_id"], semantic_vectors))

print("TF-IDF shape:", tfidf.shape)
print("LSA dimensions:", n_components)
print("Explained variance ratio:", round(float(svd.explained_variance_ratio_.sum()), 4))
print("Responses with extracted recommendation items:", int((data["recommendation_item_count"] > 0).sum()))


In [ ]:
STRUCTURE_COLUMNS = [
    "contains_bulleted_list", "contains_numbered_list", "contains_heading",
    "contains_table", "contains_code", "contains_recommendation_language",
    "contains_refusal_language", "contains_uncertainty_language", "contains_numeric_claim",
]
PRIMARY_WEIGHTS = {
    "semantic_component_scaled": 0.45,
    "lexical_component_scaled": 0.25,
    "length_component_scaled": 0.15,
    "structural_component_scaled": 0.15,
}
ALTERNATIVE_WEIGHTS = {
    "semantic_heavy": {"semantic_component_scaled": 0.60, "lexical_component_scaled": 0.20, "length_component_scaled": 0.05, "structural_component_scaled": 0.15},
    "equal_weights": {"semantic_component_scaled": 0.25, "lexical_component_scaled": 0.25, "length_component_scaled": 0.25, "structural_component_scaled": 0.25},
    "no_length": {"semantic_component_scaled": 0.55, "lexical_component_scaled": 0.25, "length_component_scaled": 0.00, "structural_component_scaled": 0.20},
}


def mean_jaccard_distances(sets, require_nonempty=False):
    n = len(sets)
    output = np.full(n, np.nan)
    for i in range(n):
        distances = []
        for j in range(n):
            if i == j:
                continue
            left, right = sets[i], sets[j]
            if require_nonempty and (not left or not right):
                continue
            union = left | right
            if union:
                distances.append(1 - len(left & right) / len(union))
        if distances:
            output[i] = float(np.mean(distances))
    return output


def compute_group_metrics(frame):
    out = frame.copy()
    metric_defaults = {
        "group_response_count": np.nan,
        "lsa_semantic_divergence": np.nan,
        "group_pairwise_semantic_distance": np.nan,
        "lexical_jaccard_instability": np.nan,
        "response_length_deviation_pct": np.nan,
        "structural_deviation": np.nan,
        "recommendation_instability_proxy": np.nan,
        "recommendation_extraction_coverage_group": np.nan,
    }
    for column, default in metric_defaults.items():
        out[column] = default

    for _, indices in out.groupby(["model_name", "base_intent_id"], observed=True).groups.items():
        idx = list(indices)
        vectors = np.vstack([vector_lookup[value] for value in out.loc[idx, "response_id"]])
        centroid = vectors.mean(axis=0, keepdims=True)
        centroid = normalize(centroid)
        centroid_distance = 1 - cosine_similarity(vectors, centroid).ravel()
        pairwise_semantic = pairwise_distances(vectors, metric="cosine")
        pairwise_row_mean = pairwise_semantic.sum(axis=1) / max(len(idx) - 1, 1)

        token_sets = out.loc[idx, "content_token_set"].tolist()
        lexical = mean_jaccard_distances(token_sets)
        recommendation_sets = [set(value) for value in out.loc[idx, "recommendation_items"]]
        recommendation = mean_jaccard_distances(recommendation_sets, require_nonempty=True)
        recommendation_coverage = np.mean([bool(value) for value in recommendation_sets])

        lengths = out.loc[idx, "response_word_count"].astype(float)
        median_length = max(float(lengths.median()), 1.0)
        length_deviation = (lengths - median_length).abs() / median_length

        structures = out.loc[idx, STRUCTURE_COLUMNS].astype(float).to_numpy()
        modal_structure = (structures.mean(axis=0) >= 0.5).astype(float)
        structural_deviation = np.mean(structures != modal_structure, axis=1)

        out.loc[idx, "group_response_count"] = len(idx)
        out.loc[idx, "lsa_semantic_divergence"] = centroid_distance
        out.loc[idx, "group_pairwise_semantic_distance"] = pairwise_row_mean
        out.loc[idx, "lexical_jaccard_instability"] = lexical
        out.loc[idx, "response_length_deviation_pct"] = length_deviation.to_numpy()
        out.loc[idx, "structural_deviation"] = structural_deviation
        out.loc[idx, "recommendation_instability_proxy"] = recommendation
        out.loc[idx, "recommendation_extraction_coverage_group"] = recommendation_coverage
    return out


full_metrics = compute_group_metrics(data)
component_source = {
    "semantic_component_scaled": "lsa_semantic_divergence",
    "lexical_component_scaled": "lexical_jaccard_instability",
    "length_component_scaled": "response_length_deviation_pct",
    "structural_component_scaled": "structural_deviation",
}
component_scales = {
    output: max(float(full_metrics[source].quantile(0.95)), 1e-9)
    for output, source in component_source.items()
}


def add_composite_scores(frame):
    out = frame.copy()
    for output, source in component_source.items():
        out[output] = (out[source] / component_scales[output]).clip(0, 1)
    out["core_behavioural_sensitivity"] = sum(out[column] * weight for column, weight in PRIMARY_WEIGHTS.items())
    for name, weights in ALTERNATIVE_WEIGHTS.items():
        out[f"core_sensitivity_{name}"] = sum(out[column] * weight for column, weight in weights.items())
    return out


full_metrics = add_composite_scores(full_metrics)
uncapped_metrics = add_composite_scores(compute_group_metrics(data.loc[~data["at_output_token_limit"]].copy()))

export_full = full_metrics.copy()
export_full["content_token_set"] = export_full["content_token_set"].map(lambda value: json.dumps(sorted(value)))
export_full["recommendation_items"] = export_full["recommendation_items"].map(json.dumps)
export_full.to_csv(TABLE_DIR / "analysis_enriched_full_observed.csv", index=False)

print("Full observed analysis rows:", len(full_metrics))
print("Uncapped robustness rows:", len(uncapped_metrics))
print("Component scaling constants:", component_scales)


## Step 6 — Exploratory data analysis


In [ ]:
descriptive_by_model = full_metrics.groupby("model_name", observed=True).agg(
    n=("response_id", "size"),
    word_count_mean=("response_word_count", "mean"),
    word_count_median=("response_word_count", "median"),
    word_count_sd=("response_word_count", "std"),
    lexical_diversity_mean=("lexical_diversity", "mean"),
    semantic_divergence_mean=("lsa_semantic_divergence", "mean"),
    lexical_instability_mean=("lexical_jaccard_instability", "mean"),
    core_sensitivity_mean=("core_behavioural_sensitivity", "mean"),
    claim_risk_proxy_mean=("claim_verification_risk_proxy", "mean"),
    recommendation_proxy_coverage=("recommendation_instability_proxy", lambda values: values.notna().mean()),
    token_ceiling_rate=("at_output_token_limit", "mean"),
).reset_index()

descriptive_by_phrasing = full_metrics.groupby("phrasing_type", observed=True).agg(
    n=("response_id", "size"),
    word_count_median=("response_word_count", "median"),
    semantic_divergence_mean=("lsa_semantic_divergence", "mean"),
    lexical_instability_mean=("lexical_jaccard_instability", "mean"),
    core_sensitivity_mean=("core_behavioural_sensitivity", "mean"),
    claim_risk_proxy_mean=("claim_verification_risk_proxy", "mean"),
    recommendation_instability_mean=("recommendation_instability_proxy", "mean"),
    recommendation_proxy_coverage=("recommendation_instability_proxy", lambda values: values.notna().mean()),
    token_ceiling_rate=("at_output_token_limit", "mean"),
).reset_index()

feature_prevalence = full_metrics.groupby("model_name", observed=True)[STRUCTURE_COLUMNS].mean().T.reset_index().rename(columns={"index": "feature"})
correlation_columns = [
    "response_word_count", "lexical_diversity", "lsa_semantic_divergence",
    "lexical_jaccard_instability", "response_length_deviation_pct", "structural_deviation",
    "recommendation_instability_proxy", "claim_verification_risk_proxy", "core_behavioural_sensitivity",
]
spearman_correlations = full_metrics[correlation_columns].corr(method="spearman")

descriptive_by_model.to_csv(TABLE_DIR / "eda_descriptive_by_model.csv", index=False)
descriptive_by_phrasing.to_csv(TABLE_DIR / "eda_descriptive_by_phrasing.csv", index=False)
feature_prevalence.to_csv(TABLE_DIR / "eda_feature_prevalence.csv", index=False)
spearman_correlations.to_csv(TABLE_DIR / "eda_spearman_correlations.csv")

display(descriptive_by_model.round(4))
display(descriptive_by_phrasing.round(4))


In [ ]:
def save_figure(name):
    path = FIGURE_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    return path


figure_paths = []

plt.figure(figsize=(8, 4.8))
coverage_plot = coverage_by_model.melt(
    id_vars="model_name", value_vars=["observed_responses", "missing_responses", "token_ceiling_responses"],
    var_name="measure", value_name="count",
)
sns.barplot(data=coverage_plot, x="model_name", y="count", hue="measure")
plt.title("Dataset coverage and collection limitations by model")
plt.xlabel("Model"); plt.ylabel("Responses"); plt.legend(title="")
figure_paths.append(save_figure("figure_01_dataset_coverage_by_model.png"))

plt.figure(figsize=(8, 4.8))
sns.boxplot(data=full_metrics, x="model_name", y="response_word_count", order=MODEL_ORDER, showfliers=False)
plt.title("Response word-count distribution by model")
plt.xlabel("Model"); plt.ylabel("Response word count")
figure_paths.append(save_figure("figure_02_response_word_count_by_model.png"))

ceiling_heatmap = coverage_by_phrasing.pivot(index="model_name", columns="phrasing_type", values="token_ceiling_rate").reindex(index=MODEL_ORDER, columns=PHRASING_ORDER)
plt.figure(figsize=(14, 3.8))
sns.heatmap(ceiling_heatmap, annot=True, fmt=".0%", cmap="Reds", vmin=0, vmax=1, cbar_kws={"label": "Token-ceiling rate"})
plt.title("1,200-token ceiling rate by model and phrasing")
plt.xlabel("Phrasing type"); plt.ylabel("Model"); plt.xticks(rotation=35, ha="right")
figure_paths.append(save_figure("figure_03_token_ceiling_heatmap.png"))

feature_labels = {
    "contains_bulleted_list": "Bulleted list", "contains_numbered_list": "Numbered list",
    "contains_heading": "Heading", "contains_table": "Table", "contains_code": "Code",
    "contains_recommendation_language": "Recommendation language",
    "contains_refusal_language": "Refusal language", "contains_uncertainty_language": "Uncertainty language",
    "contains_numeric_claim": "Numeric claim",
}
feature_long = feature_prevalence.melt(id_vars="feature", var_name="model_name", value_name="proportion")
feature_long["feature_label"] = feature_long["feature"].map(feature_labels).fillna(feature_long["feature"])
plt.figure(figsize=(11, 5.5))
sns.barplot(data=feature_long, x="feature_label", y="proportion", hue="model_name", hue_order=MODEL_ORDER)
plt.title("Response-structure feature prevalence")
plt.xlabel(""); plt.ylabel("Proportion of responses"); plt.xticks(rotation=35, ha="right"); plt.legend(title="Model")
figure_paths.append(save_figure("figure_04_response_structure_prevalence.png"))

measure_labels = {
    "response_word_count": "Word count", "lexical_diversity": "Lexical diversity",
    "lsa_semantic_divergence": "Semantic divergence", "lexical_jaccard_instability": "Lexical instability",
    "response_length_deviation_pct": "Length deviation", "structural_deviation": "Structural deviation",
    "recommendation_instability_proxy": "Recommendation instability proxy",
    "claim_verification_risk_proxy": "Claim-verification risk proxy",
    "core_behavioural_sensitivity": "Behavioural sensitivity",
}
correlation_plot = spearman_correlations.rename(index=measure_labels, columns=measure_labels)
plt.figure(figsize=(8, 7))
sns.heatmap(correlation_plot, cmap="vlag", center=0, vmin=-1, vmax=1, annot=True, fmt=".2f", square=True)
plt.title("Spearman correlations among response measures")
figure_paths.append(save_figure("figure_05_spearman_correlation_heatmap.png"))

print("EDA figures created:", len(figure_paths))


## Step 7 — Statistical helpers

The 100 base intents are the paired blocks. Friedman tests assess overall differences across repeated phrasing/model conditions. Kendall's W is reported as the omnibus effect size. Significant omnibus tests are followed by paired Wilcoxon tests with Holm correction and matched-pairs rank-biserial effect sizes.


In [ ]:
def holm_adjust(p_values):
    values = np.asarray(p_values, dtype=float)
    order = np.argsort(values)
    adjusted = np.empty_like(values)
    running = 0.0
    m = len(values)
    for rank, position in enumerate(order):
        candidate = (m - rank) * values[position]
        running = max(running, candidate)
        adjusted[position] = min(running, 1.0)
    return adjusted


def rank_biserial_paired(differences):
    differences = np.asarray(differences, dtype=float)
    differences = differences[np.isfinite(differences) & (differences != 0)]
    if len(differences) == 0:
        return 0.0
    ranks = stats.rankdata(np.abs(differences))
    positive = ranks[differences > 0].sum()
    negative = ranks[differences < 0].sum()
    return float((positive - negative) / (positive + negative))


def friedman_test(frame, block, condition, value, condition_order, analysis, model=None, minimum_blocks=10):
    pivot = frame.pivot_table(index=block, columns=condition, values=value, aggfunc="mean")
    available = [item for item in condition_order if item in pivot.columns]
    pivot = pivot.reindex(columns=available).dropna()
    n, k = pivot.shape
    if n < minimum_blocks or k < 3:
        return None, pivot
    arrays = [pivot[column].to_numpy() for column in available]
    if np.allclose(np.nanstd(pivot.to_numpy(), axis=1), 0):
        statistic, p_value = 0.0, 1.0
    else:
        statistic, p_value = stats.friedmanchisquare(*arrays)
    effect = float(statistic / (n * (k - 1))) if n and k > 1 else np.nan
    result = {
        "analysis": analysis, "model_name": model or "All", "outcome": value,
        "n_blocks": n, "n_conditions": k, "friedman_chi2": float(statistic),
        "degrees_of_freedom": k - 1, "p_value": float(p_value), "kendalls_w": effect,
    }
    return result, pivot


def pairwise_wilcoxon(pivot, analysis, model, outcome):
    rows = []
    for left, right in combinations(pivot.columns, 2):
        left_values = pivot[left].to_numpy(dtype=float)
        right_values = pivot[right].to_numpy(dtype=float)
        differences = left_values - right_values
        if np.allclose(differences, 0):
            statistic, p_value = 0.0, 1.0
        else:
            statistic, p_value = stats.wilcoxon(left_values, right_values, zero_method="wilcox", alternative="two-sided", method="auto")
        rows.append({
            "analysis": analysis, "model_name": model, "outcome": outcome,
            "condition_a": str(left), "condition_b": str(right), "n_pairs": len(differences),
            "median_a": float(np.median(left_values)), "median_b": float(np.median(right_values)),
            "median_difference_a_minus_b": float(np.median(differences)),
            "wilcoxon_statistic": float(statistic), "p_value_raw": float(p_value),
            "rank_biserial_a_minus_b": rank_biserial_paired(differences),
        })
    result = pd.DataFrame(rows)
    if not result.empty:
        result["p_value_holm"] = holm_adjust(result["p_value_raw"])
        result["significant_holm"] = result["p_value_holm"] < ALPHA
    return result


def run_repeated_analysis(frame, outcomes, by_model=True, analysis_prefix=""):
    omnibus_rows, pairwise_frames = [], []
    model_values = MODEL_ORDER if by_model else [None]
    for model in model_values:
        subset = frame.loc[frame["model_name"].astype(str).eq(model)].copy() if model else frame.copy()
        for outcome in outcomes:
            result, pivot = friedman_test(
                subset, block="base_intent_id", condition="phrasing_type", value=outcome,
                condition_order=PHRASING_ORDER, analysis=analysis_prefix, model=model,
            )
            if result is None:
                continue
            omnibus_rows.append(result)
            if result["p_value"] < ALPHA:
                pairwise_frames.append(pairwise_wilcoxon(pivot, analysis_prefix, model or "All", outcome))
    omnibus = pd.DataFrame(omnibus_rows)
    pairwise = pd.concat(pairwise_frames, ignore_index=True) if pairwise_frames else pd.DataFrame()
    return omnibus, pairwise


## RQ1 — Do equivalent wording changes affect behavioural consistency?


In [ ]:
rq1_outcomes = ["core_behavioural_sensitivity", "lsa_semantic_divergence", "lexical_jaccard_instability"]
rq1_omnibus, rq1_pairwise = run_repeated_analysis(
    full_metrics, rq1_outcomes, by_model=True, analysis_prefix="RQ1_full_observed"
)
rq1_omnibus.to_csv(TABLE_DIR / "rq1_friedman_tests.csv", index=False)
rq1_pairwise.to_csv(TABLE_DIR / "rq1_pairwise_wilcoxon_holm.csv", index=False)

rq1_descriptives = full_metrics.groupby(["model_name", "phrasing_type"], observed=True)[rq1_outcomes].agg(["count", "mean", "median", "std"]).reset_index()
rq1_descriptives.columns = ["_".join([str(part) for part in column if part]).rstrip("_") if isinstance(column, tuple) else column for column in rq1_descriptives.columns]
rq1_descriptives.to_csv(TABLE_DIR / "rq1_descriptives_by_model_phrasing.csv", index=False)

display(rq1_omnibus.round(6))
if not rq1_pairwise.empty:
    display(rq1_pairwise.loc[rq1_pairwise["significant_holm"]].sort_values("p_value_holm").head(20).round(6))

plt.figure(figsize=(14, 5.5))
sns.pointplot(
    data=full_metrics, x="phrasing_type", y="core_behavioural_sensitivity", hue="model_name",
    order=PHRASING_ORDER, hue_order=MODEL_ORDER, errorbar=("ci", 95), dodge=0.25, markers="o",
)
plt.title("RQ1: Behavioural sensitivity by phrasing and model")
plt.xlabel("Phrasing type"); plt.ylabel("Mean core behavioural sensitivity (95% CI)")
plt.xticks(rotation=35, ha="right"); plt.legend(title="Model")
figure_paths.append(save_figure("figure_06_rq1_core_sensitivity_by_phrasing_model.png"))


## RQ2 — Which linguistic structures most affect the three behavioural dimensions?

The three outcomes are analysed separately. The claim-risk measure is a verification-priority proxy, and the recommendation result must always be accompanied by extraction coverage.


In [ ]:
rq2_outcomes = [
    "lsa_semantic_divergence", "claim_verification_risk_proxy", "recommendation_instability_proxy"
]
rq2_omnibus, rq2_pairwise = run_repeated_analysis(
    full_metrics, rq2_outcomes, by_model=True, analysis_prefix="RQ2_full_observed"
)
rq2_omnibus.to_csv(TABLE_DIR / "rq2_friedman_tests.csv", index=False)
rq2_pairwise.to_csv(TABLE_DIR / "rq2_pairwise_wilcoxon_holm.csv", index=False)

rq2_summary = full_metrics.groupby(["model_name", "phrasing_type"], observed=True).agg(
    n=("response_id", "size"),
    semantic_divergence_mean=("lsa_semantic_divergence", "mean"),
    semantic_divergence_median=("lsa_semantic_divergence", "median"),
    claim_risk_proxy_mean=("claim_verification_risk_proxy", "mean"),
    recommendation_instability_mean=("recommendation_instability_proxy", "mean"),
    recommendation_proxy_n=("recommendation_instability_proxy", "count"),
    token_ceiling_rate=("at_output_token_limit", "mean"),
).reset_index()
rq2_summary["recommendation_proxy_coverage"] = rq2_summary["recommendation_proxy_n"] / rq2_summary["n"]
rq2_summary.to_csv(TABLE_DIR / "rq2_dimension_summary_by_model_phrasing.csv", index=False)

display(rq2_omnibus.round(6))

rq2_plot_specs = [
    ("lsa_semantic_divergence", "Latent-semantic divergence", "figure_07_rq2_semantic_divergence.png"),
    ("claim_verification_risk_proxy", "Claim-verification risk proxy", "figure_08_rq2_claim_risk_proxy.png"),
    ("recommendation_instability_proxy", "Recommendation instability proxy", "figure_09_rq2_recommendation_instability.png"),
]
for outcome, label, filename in rq2_plot_specs:
    plot_data = full_metrics.loc[full_metrics[outcome].notna()].copy()
    if plot_data.empty:
        continue
    plt.figure(figsize=(14, 5.2))
    sns.pointplot(
        data=plot_data, x="phrasing_type", y=outcome, hue="model_name",
        order=PHRASING_ORDER, hue_order=MODEL_ORDER, errorbar=("ci", 95), dodge=0.25,
    )
    plt.title(f"RQ2: {label} by phrasing and model")
    plt.xlabel("Phrasing type"); plt.ylabel(f"Mean {label.lower()} (95% CI)")
    plt.xticks(rotation=35, ha="right"); plt.legend(title="Model")
    figure_paths.append(save_figure(filename))


## RQ3 — Does wording sensitivity differ across models?


In [ ]:
group_level = full_metrics.groupby(["base_intent_id", "domain", "model_name"], observed=True).agg(
    group_response_count=("response_id", "size"),
    group_core_sensitivity=("core_behavioural_sensitivity", "mean"),
    group_semantic_sensitivity=("group_pairwise_semantic_distance", "mean"),
    group_claim_risk_proxy=("claim_verification_risk_proxy", "mean"),
    group_recommendation_instability=("recommendation_instability_proxy", "mean"),
    group_token_ceiling_rate=("at_output_token_limit", "mean"),
).reset_index()
group_level.to_csv(TABLE_DIR / "rq3_group_level_sensitivity.csv", index=False)

rq3_rows, rq3_pairwise_frames = [], []
for outcome in ["group_core_sensitivity", "group_semantic_sensitivity"]:
    result, pivot = friedman_test(
        group_level, block="base_intent_id", condition="model_name", value=outcome,
        condition_order=MODEL_ORDER, analysis="RQ3_full_observed", model="All_models",
    )
    if result:
        rq3_rows.append(result)
        if result["p_value"] < ALPHA:
            rq3_pairwise_frames.append(pairwise_wilcoxon(pivot, "RQ3_full_observed", "All_models", outcome))
rq3_omnibus = pd.DataFrame(rq3_rows)
rq3_pairwise = pd.concat(rq3_pairwise_frames, ignore_index=True) if rq3_pairwise_frames else pd.DataFrame()
rq3_omnibus.to_csv(TABLE_DIR / "rq3_friedman_model_comparison.csv", index=False)
rq3_pairwise.to_csv(TABLE_DIR / "rq3_pairwise_models_wilcoxon_holm.csv", index=False)

display(rq3_omnibus.round(6))
if not rq3_pairwise.empty:
    display(rq3_pairwise.round(6))

plt.figure(figsize=(8, 5))
sns.boxplot(data=group_level, x="model_name", y="group_core_sensitivity", order=MODEL_ORDER, showfliers=False)
sns.stripplot(data=group_level, x="model_name", y="group_core_sensitivity", order=MODEL_ORDER, color="black", alpha=0.25, size=2.5)
plt.title("RQ3: Within-intent behavioural sensitivity by model")
plt.xlabel("Model"); plt.ylabel("Group mean core behavioural sensitivity")
figure_paths.append(save_figure("figure_10_rq3_group_sensitivity_by_model.png"))


## Step 8 — Robustness and collection-limit sensitivity analyses

Three checks are used:

1. recompute all group metrics after removing token-ceiling responses;
2. compare models on the same prompt IDs where all three produced uncapped responses;
3. repeat the group-level model comparison under alternative composite weights, including a no-length score.

Divergence between the full and uncapped results is evidence that collection censoring affects the conclusion and must be discussed.


In [ ]:
robustness_rows = []

rq1_uncapped, _ = run_repeated_analysis(
    uncapped_metrics, ["core_behavioural_sensitivity", "lsa_semantic_divergence"],
    by_model=True, analysis_prefix="RQ1_uncapped_only",
)
if not rq1_uncapped.empty:
    robustness_rows.extend(rq1_uncapped.to_dict(orient="records"))

uncapped_group_level = uncapped_metrics.groupby(["base_intent_id", "model_name"], observed=True).agg(
    group_response_count=("response_id", "size"),
    group_core_sensitivity=("core_behavioural_sensitivity", "mean"),
    group_semantic_sensitivity=("group_pairwise_semantic_distance", "mean"),
).reset_index()
uncapped_group_level = uncapped_group_level.loc[uncapped_group_level["group_response_count"] >= 3]
for outcome in ["group_core_sensitivity", "group_semantic_sensitivity"]:
    result, _ = friedman_test(
        uncapped_group_level, block="base_intent_id", condition="model_name", value=outcome,
        condition_order=MODEL_ORDER, analysis="RQ3_uncapped_minimum_3_per_group", model="All_models",
    )
    if result:
        robustness_rows.append(result)

uncapped_model_counts = data.loc[~data["at_output_token_limit"]].groupby("prompt_id", observed=True)["model_name"].nunique()
common_uncapped_prompts = uncapped_model_counts.loc[uncapped_model_counts.eq(3)].index
common_subset = full_metrics.loc[
    full_metrics["prompt_id"].isin(common_uncapped_prompts) & ~full_metrics["at_output_token_limit"]
].copy()
common_prompt_rows, common_prompt_pairwise = [], []
for outcome in ["response_word_count", "lexical_diversity", "claim_verification_risk_proxy"]:
    result, pivot = friedman_test(
        common_subset, block="prompt_id", condition="model_name", value=outcome,
        condition_order=MODEL_ORDER, analysis="Common_uncapped_prompt_model_comparison", model="All_models",
        minimum_blocks=30,
    )
    if result:
        common_prompt_rows.append(result)
        if result["p_value"] < ALPHA:
            common_prompt_pairwise.append(pairwise_wilcoxon(pivot, "Common_uncapped_prompt_model_comparison", "All_models", outcome))

alternative_group_results = []
for score_name in ["core_sensitivity_semantic_heavy", "core_sensitivity_equal_weights", "core_sensitivity_no_length"]:
    alternative = full_metrics.groupby(["base_intent_id", "model_name"], observed=True)[score_name].mean().reset_index()
    result, _ = friedman_test(
        alternative, block="base_intent_id", condition="model_name", value=score_name,
        condition_order=MODEL_ORDER, analysis="RQ3_alternative_composite_weights", model="All_models",
    )
    if result:
        alternative_group_results.append(result)

robustness_omnibus = pd.DataFrame(robustness_rows + common_prompt_rows + alternative_group_results)
robustness_pairwise = pd.concat(common_prompt_pairwise, ignore_index=True) if common_prompt_pairwise else pd.DataFrame()
robustness_omnibus.to_csv(TABLE_DIR / "robustness_omnibus_tests.csv", index=False)
robustness_pairwise.to_csv(TABLE_DIR / "robustness_common_prompt_pairwise.csv", index=False)
uncapped_group_level.to_csv(TABLE_DIR / "robustness_uncapped_group_level.csv", index=False)
common_subset.to_csv(TABLE_DIR / "robustness_common_uncapped_prompt_rows.csv", index=False)

robustness_summary = pd.DataFrame([
    {"scenario": "Full observed outputs", "rows": len(full_metrics), "unique_prompts": full_metrics["prompt_id"].nunique(), "notes": "Includes token-ceiling outputs as realised responses"},
    {"scenario": "Uncapped-only", "rows": len(uncapped_metrics), "unique_prompts": uncapped_metrics["prompt_id"].nunique(), "notes": "Metrics recomputed after excluding ceiling responses"},
    {"scenario": "Common uncapped prompt IDs across all models", "rows": len(common_subset), "unique_prompts": len(common_uncapped_prompts), "notes": "Direct model comparison on matched prompt IDs"},
])
robustness_summary.to_csv(TABLE_DIR / "robustness_sample_summary.csv", index=False)
display(robustness_summary)
display(robustness_omnibus.round(6))

alternative_plot = full_metrics.groupby("model_name", observed=True)[
    ["core_behavioural_sensitivity", "core_sensitivity_semantic_heavy", "core_sensitivity_equal_weights", "core_sensitivity_no_length"]
].mean().reset_index().melt(id_vars="model_name", var_name="weighting", value_name="mean_score")
weighting_labels = {
    "core_behavioural_sensitivity": "Primary weights", "core_sensitivity_semantic_heavy": "Semantic-heavy",
    "core_sensitivity_equal_weights": "Equal weights", "core_sensitivity_no_length": "No length component",
}
alternative_plot["weighting"] = alternative_plot["weighting"].map(weighting_labels)
plt.figure(figsize=(10, 5))
sns.barplot(data=alternative_plot, x="model_name", y="mean_score", hue="weighting", order=MODEL_ORDER)
plt.title("Robustness of model sensitivity under alternative composite weights")
plt.xlabel("Model"); plt.ylabel("Mean sensitivity score"); plt.legend(title="Weighting", fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
figure_paths.append(save_figure("figure_11_robustness_alternative_weights.png"))


## Step 9 — Build the Linguistic Influence Matrix


In [ ]:
matrix_dimensions = [
    "lsa_semantic_divergence", "lexical_jaccard_instability", "response_length_deviation_pct",
    "structural_deviation", "recommendation_instability_proxy", "claim_verification_risk_proxy",
    "core_behavioural_sensitivity",
]
influence_matrix_raw = full_metrics.groupby("phrasing_type", observed=True)[matrix_dimensions].mean().reindex(PHRASING_ORDER)
influence_matrix_z = influence_matrix_raw.apply(
    lambda column: (column - column.mean()) / column.std(ddof=0) if column.std(ddof=0) > 0 else 0
)
influence_matrix_by_model = full_metrics.groupby(["model_name", "phrasing_type"], observed=True)[matrix_dimensions].mean().reset_index()

influence_matrix_raw.to_csv(TABLE_DIR / "linguistic_influence_matrix_raw_means.csv")
influence_matrix_z.to_csv(TABLE_DIR / "linguistic_influence_matrix_standardised_z_scores.csv")
influence_matrix_by_model.to_csv(TABLE_DIR / "linguistic_influence_matrix_by_model.csv", index=False)

display(influence_matrix_raw.round(4))

matrix_labels = {
    "lsa_semantic_divergence": "Semantic divergence", "lexical_jaccard_instability": "Lexical instability",
    "response_length_deviation_pct": "Length deviation", "structural_deviation": "Structural deviation",
    "recommendation_instability_proxy": "Recommendation instability proxy",
    "claim_verification_risk_proxy": "Claim-verification risk proxy",
    "core_behavioural_sensitivity": "Behavioural sensitivity",
}
influence_matrix_plot = influence_matrix_z.rename(columns=matrix_labels)
plt.figure(figsize=(11, 6.2))
sns.heatmap(influence_matrix_plot, cmap="vlag", center=0, annot=True, fmt=".2f", cbar_kws={"label": "Standardised influence (z-score)"})
plt.title("Linguistic Influence Matrix")
plt.xlabel("Behavioural dimension"); plt.ylabel("Phrasing type")
plt.xticks(rotation=35, ha="right")
figure_paths.append(save_figure("figure_12_linguistic_influence_matrix.png"))


## Step 10 — Create a stratified manual-validation sample

Automated claim-risk cannot establish factual hallucination. This file provides a reproducible sample for manual verification, stratified across model, domain and token-ceiling status. Complete the blank coding fields using a documented protocol and, ideally, a second coder for a subset so agreement can be reported.


In [ ]:
sampled_frames = []
for _, group in full_metrics.groupby(["model_name", "domain", "at_output_token_limit"], observed=True):
    sampled_frames.append(group.sample(n=min(5, len(group)), random_state=RANDOM_SEED))
manual_sample = pd.concat(sampled_frames, ignore_index=True).drop_duplicates("response_id")
if len(manual_sample) < 150:
    remaining = full_metrics.loc[~full_metrics["response_id"].isin(manual_sample["response_id"])]
    extra_n = min(150 - len(manual_sample), len(remaining))
    manual_sample = pd.concat([manual_sample, remaining.sample(extra_n, random_state=RANDOM_SEED)], ignore_index=True)
manual_sample = manual_sample.head(150).copy()

manual_columns = [
    "response_id", "prompt_id", "base_intent_id", "domain", "intent_type", "phrasing_type",
    "model_name", "model_version", "prompt_text", "response_text", "at_output_token_limit",
    "claim_verification_risk_proxy", "contains_numeric_claim", "has_url",
]
manual_validation = manual_sample[manual_columns].copy()
manual_validation["factual_claim_present_0_1"] = ""
manual_validation["claim_supported_0_1_not_applicable"] = ""
manual_validation["inaccuracy_or_fabrication_0_1"] = ""
manual_validation["severity_0_to_3"] = ""
manual_validation["verification_source"] = ""
manual_validation["coder_notes"] = ""
manual_validation.to_csv(VALIDATION_DIR / "manual_hallucination_validation_sample.csv", index=False)

protocol = (
    "# Manual validation protocol\n\n"
    "1. Review the prompt and full response without looking at the automated proxy score first.\n"
    "2. Mark whether a checkable factual claim is present.\n"
    "3. Verify material claims using authoritative or primary sources where possible.\n"
    "4. Mark support, inaccuracy/fabrication and severity using the fixed coding columns.\n"
    "5. Record the verification source and a short rationale.\n"
    "6. Have a second coder independently review at least 20% of the sample; report agreement "
    "(Cohen's kappa for binary fields, weighted kappa for severity).\n"
    "7. Use the validated rate as the hallucination outcome. Until this is completed, call the "
    "automated measure a claim-verification risk proxy only.\n"
)
(VALIDATION_DIR / "manual_validation_protocol.md").write_text(protocol, encoding="utf-8")
print("Manual validation sample rows:", len(manual_validation))


## Step 11 — Generate the dissertation reporting guide and package


In [ ]:
def p_text(value):
    if pd.isna(value):
        return "not available"
    return "< .001" if value < 0.001 else f"= {value:.3f}"


def strongest_phrasing(frame, outcome):
    summary = frame.groupby("phrasing_type", observed=True)[outcome].mean().sort_values(ascending=False)
    return str(summary.index[0]), float(summary.iloc[0])


top_core_phrasing, top_core_value = strongest_phrasing(full_metrics, "core_behavioural_sensitivity")
top_semantic_phrasing, top_semantic_value = strongest_phrasing(full_metrics, "lsa_semantic_divergence")
top_risk_phrasing, top_risk_value = strongest_phrasing(full_metrics, "claim_verification_risk_proxy")

guide_lines = [
    "# Dissertation analysis and results guide",
    "",
    "## Dataset statement",
    f"The analysis contains {len(full_metrics):,} observed prompt-model responses covering {data['prompt_id'].nunique():,} prompts, {data['base_intent_id'].nunique()} base intents, three models and five domains.",
    f"Sixteen responses are missing from the expected 3,000-row design. {int(data['at_output_token_limit'].sum()):,} observed responses reached the fixed 1,200-token ceiling.",
    "The primary analysis treats capped text as the realised response under the collection protocol; uncapped-only and matched-prompt analyses test sensitivity to that decision.",
    "",
    "## Required methodology corrections to the current draft",
    "- Report three models, not four or more.",
    "- Report five domains: Technology, Finance, Healthcare, Marketing and Sports.",
    "- Report one run per prompt; do not claim repeated stochastic trials or inter-run agreement.",
    "- Do not state that every full response was captured; report missingness and the 1,200-token ceiling.",
    "- Describe automated hallucination results as a claim-verification risk proxy unless the manual validation sample is coded.",
    "- Describe recommendation drift as extraction-based and report its coverage.",
    "",
    "## EDA headline",
    f"The highest mean core sensitivity occurred for {top_core_phrasing} ({top_core_value:.3f}); the highest mean latent-semantic divergence occurred for {top_semantic_phrasing} ({top_semantic_value:.3f}).",
    f"The highest mean automated claim-verification risk proxy occurred for {top_risk_phrasing} ({top_risk_value:.3f}). These rankings are descriptive and should be paired with the inferential tables.",
    "",
    "## RQ1 reporting order",
    "1. State the complete-block sample size separately for each model.",
    "2. Report the Friedman chi-square, degrees of freedom, exact p-value and Kendall's W.",
    "3. Report only Holm-significant paired comparisons, including direction and rank-biserial effect size.",
    "4. Compare the full-observed and uncapped-only conclusions.",
    "",
    "## RQ2 reporting order",
    "Report semantic divergence, claim-verification risk and recommendation instability separately. Do not treat the proxy measures as ground truth. Give recommendation extraction coverage and incorporate the manually validated hallucination rate when coding is complete.",
    "",
    "## RQ3 reporting order",
    "Use the 100 base intents as paired blocks for model-level group sensitivity. Explicitly discuss differential censoring: ChatGPT reached the ceiling much more frequently than Claude, while Gemini did not reach it in this collection.",
    "",
    "## Robustness interpretation",
    f"The matched uncapped cross-model subset contains {len(common_uncapped_prompts):,} prompt IDs ({len(common_subset):,} rows). If rankings or significance change across full, uncapped and no-length analyses, present the finding as collection-condition dependent.",
    "",
    "## Distinction-level checklist",
    "- Link every table and figure to a research question rather than listing outputs.",
    "- Report effect sizes and uncertainty, not p-values alone.",
    "- Explain why paired tests use base intents as blocks.",
    "- Separate descriptive patterns, inferential evidence and interpretation.",
    "- Critically compare results with the literature in the Discussion chapter.",
    "- Include organisational implications for prompt governance, testing and model selection.",
    "- State construct-validity limits of each proxy and avoid causal or universal claims.",
    "",
    "## Output index",
    "The `tables` folder contains data-quality, EDA, RQ1-RQ3, robustness and Linguistic Influence Matrix tables. The `figures` folder contains publication-ready PNGs. The `manual_validation` folder contains the validation sample and coding protocol.",
]

if not rq1_omnibus.empty:
    guide_lines.extend(["", "## RQ1 omnibus results generated by this run"])
    for row in rq1_omnibus.itertuples():
        guide_lines.append(
            f"- {row.model_name}, {row.outcome}: χ²({row.degrees_of_freedom}) = {row.friedman_chi2:.3f}, "
            f"p {p_text(row.p_value)}, Kendall's W = {row.kendalls_w:.3f}, n = {row.n_blocks} base intents."
        )
if not rq3_omnibus.empty:
    guide_lines.extend(["", "## RQ3 omnibus results generated by this run"])
    for row in rq3_omnibus.itertuples():
        guide_lines.append(
            f"- {row.outcome}: χ²({row.degrees_of_freedom}) = {row.friedman_chi2:.3f}, "
            f"p {p_text(row.p_value)}, Kendall's W = {row.kendalls_w:.3f}, n = {row.n_blocks} base intents."
        )

results_guide_path = OUTPUT_DIR / "RESULTS_AND_WRITEUP_GUIDE.md"
results_guide_path.write_text("\n".join(guide_lines) + "\n", encoding="utf-8")

analysis_summary = {
    "input_file": INPUT_FILE.name,
    "rows": int(len(full_metrics)),
    "expected_rows": 3000,
    "missing_responses": int(expected_grid["missing_response"].sum()),
    "token_ceiling_responses": int(data["at_output_token_limit"].sum()),
    "models": MODEL_ORDER,
    "domains": DOMAIN_ORDER,
    "base_intents": int(data["base_intent_id"].nunique()),
    "prompts": int(data["prompt_id"].nunique()),
    "common_uncapped_prompt_ids_all_models": int(len(common_uncapped_prompts)),
    "component_scales": component_scales,
    "primary_weights": PRIMARY_WEIGHTS,
    "random_seed": RANDOM_SEED,
    "alpha": ALPHA,
    "figures_created": len(figure_paths),
}
summary_path = OUTPUT_DIR / "analysis_summary.json"
summary_path.write_text(json.dumps(analysis_summary, indent=2), encoding="utf-8")

package_path = OUTPUT_DIR / "dissertation_analysis_package.zip"
with zipfile.ZipFile(package_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file() and path != package_path:
            archive.write(path, arcname=path.relative_to(OUTPUT_DIR))

print(json.dumps(analysis_summary, indent=2))
print("Package:", package_path)


## Step 12 — Final reproducibility and output verification


In [ ]:
required_outputs = [
    TABLE_DIR / "data_quality_summary.csv",
    TABLE_DIR / "eda_descriptive_by_model.csv",
    TABLE_DIR / "rq1_friedman_tests.csv",
    TABLE_DIR / "rq2_friedman_tests.csv",
    TABLE_DIR / "rq3_friedman_model_comparison.csv",
    TABLE_DIR / "robustness_omnibus_tests.csv",
    TABLE_DIR / "linguistic_influence_matrix_raw_means.csv",
    FIGURE_DIR / "figure_12_linguistic_influence_matrix.png",
    VALIDATION_DIR / "manual_hallucination_validation_sample.csv",
    OUTPUT_DIR / "RESULTS_AND_WRITEUP_GUIDE.md",
    OUTPUT_DIR / "analysis_summary.json",
    OUTPUT_DIR / "dissertation_analysis_package.zip",
]
missing_outputs = [str(path) for path in required_outputs if not path.exists() or path.stat().st_size == 0]
if missing_outputs:
    raise FileNotFoundError(f"Missing or empty outputs: {missing_outputs}")

reloaded_rq1 = pd.read_csv(TABLE_DIR / "rq1_friedman_tests.csv")
reloaded_rq3 = pd.read_csv(TABLE_DIR / "rq3_friedman_model_comparison.csv")
reloaded_matrix = pd.read_csv(TABLE_DIR / "linguistic_influence_matrix_raw_means.csv")
assert len(reloaded_rq1) >= 3
assert len(reloaded_rq3) >= 1
assert len(reloaded_matrix) == len(PHRASING_ORDER)
assert all(path.exists() and path.stat().st_size > 20_000 for path in figure_paths)

print("All required outputs passed verification.")
print("Figures:", len(figure_paths))
print("Tables:", len(list(TABLE_DIR.glob("*.csv"))))
print("Analysis package:", package_path)

if IN_COLAB:
    from google.colab import files
    files.download(str(package_path))


## What to do after running this notebook

1. Read `RESULTS_AND_WRITEUP_GUIDE.md` alongside the RQ tables.
2. Inspect every figure; keep only those that directly support a research question.
3. Complete the manual hallucination-validation sample before making factual hallucination claims.
4. Update the Methodology chapter so it describes the realised three-model, five-domain, single-run design.
5. Write the Results chapter from the statistical tables, then interpret those findings against the literature in the Discussion chapter.
6. Retain the quality and robustness tables in the appendix even if they are not all shown in the main chapter.
